In [ ]:
!git clone https://github.com/apffkxhsls/OSS-Blind-Walk-Assistant.git
%cd OSS-Blind-Walk-Assistant

In [ ]:
!pip install ultralytics roboflow

In [ ]:
# Roboflow에서 v2 데이터셋 다운로드
from roboflow import Roboflow
rf = Roboflow(api_key="3ZPPnYZwsKJGOAMCoOYY")
project = rf.workspace("s-workspace-1cqyr").project("oss-blind-walk-assistant")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
# YOLO11n 모델 학습 (v2)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data=dataset.location + "/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="blindwalk_v2"
)

In [ ]:
!ls /content/OSS-Blind-Walk-Assistant/runs/detect

In [ ]:
import os, glob, shutil
import pandas as pd

# 1. 이번 학습 결과 폴더 이름
run_dir = "/content/OSS-Blind-Walk-Assistant/runs/detect/blindwalk_v2"

# 2. 저장할 버전 폴더
save_dir = "/content/OSS-Blind-Walk-Assistant/results/v2_IrregularDatasetSize"
os.makedirs(save_dir, exist_ok=True)

# 3. results.png, confusion_matrix.png 복사
for file in ["results.png", "confusion_matrix.png"]:
    src = os.path.join(run_dir, file)
    if os.path.exists(src):
        shutil.copy(src, save_dir)

# 4. val_batch*_pred.jpg 복사
for src in glob.glob(os.path.join(run_dir, "val_batch*_pred.jpg")):
    shutil.copy(src, save_dir)

print("복사 완료:", save_dir)

In [ ]:
# version에 따른 클래스 확인
import yaml

with open(dataset.location + "/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print("Classes:", len(data["names"]))
print(data["names"])

In [ ]:
# 성능 분석에 필요한 수치값 확인
import pandas as pd

df = pd.read_csv("/content/OSS-Blind-Walk-Assistant/runs/detect/blindwalk_v2/results.csv")

print("Precision:", df.iloc[-1]["metrics/precision(B)"])
print("Recall:", df.iloc[-1]["metrics/recall(B)"])
print("mAP50:", df.iloc[-1]["metrics/mAP50(B)"])
print("mAP50-95:", df.iloc[-1]["metrics/mAP50-95(B)"])

In [ ]:
metrics_text = """
Version: v2_IrregularDatasetSize

Model: YOLO11n
Epochs: 50
Image Size: 640
Batch Size: 8

Dataset
- Total Images: 408
- Classes: 10

Class List
- bicycle
- bollard
- car
- damaged_braille_block
- fire_hydrant
- kickboard
- motorcycle
- traffic_cone
- trash
- utility_pole

Metrics
Precision: 0.69468
Recall: 0.44115
mAP50: 0.51363
mAP50-95: 0.34802
"""

with open("/content/OSS-Blind-Walk-Assistant/results/v2_IrregularDatasetSize/metrics.txt", "w") as f:
    f.write(metrics_text)

print("metrics.txt 저장 완료!")

In [ ]:
import os

folder = "/content/OSS-Blind-Walk-Assistant/results/v2_IrregularDatasetSize"

for f in sorted(os.listdir(folder)):
    print(f)

In [ ]:
!cat /content/OSS-Blind-Walk-Assistant/results/v2_IrregularDatasetSize/metrics.txt

In [ ]:
!zip -r v2_IrregularDatasetSize.zip /content/OSS-Blind-Walk-Assistant/results/v2_IrregularDatasetSize

In [ ]:
from google.colab import files
files.download("v2_IrregularDatasetSize.zip")

In [ ]:
from google.colab import files
files.download("/content/OSS-Blind-Walk-Assistant/runs/detect/blindwalk_v2/weights/best.pt")